<a href="https://colab.research.google.com/github/Heba2627/NLP-Project/blob/main/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing Dataset via. Pandas

In [86]:
import pandas as pd
import numpy as np
import regex as re

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}
dfs=[]
for splitname,path in splits.items():
    df=pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/"+ path)
    df["split"]=splitname
    dfs.append(df)

df=pd.concat(dfs, ignore_index=True)

In [87]:
df=df[df["lang"].isin(["ar", "ko", "te"])]

In [88]:
print(df.columns)
answerable_df = df[df["answerable"] == True]
unanswerable_df = df[df["answerable"] == False]

print(f"Number of answerable instances: {len(answerable_df)}")
print(f"Number of unanswerable instances: {len(unanswerable_df)}")
print(f"{len(answerable_df)/len(df)*100:.2f}% of dataset that is answerable")
print(f"{len(unanswerable_df)/len(df)*100:.2f}% of dataset that is unanswerable")

Index(['question', 'context', 'lang', 'answerable', 'answer_start', 'answer',
       'answer_inlang', 'split'],
      dtype='str')
Number of answerable instances: 6963
Number of unanswerable instances: 527
92.96% of dataset that is answerable
7.04% of dataset that is unanswerable


In [89]:
#Check for letters in all languages + Mark for arabic
train_df= df[df["split"] == "train"]
val_df= df[df["split"] == "validation"]

word_pattern = re.compile(r"[\p{L}\p{M}]+", flags=re.UNICODE)

def n_tokens(text):
    return len(word_pattern.findall(str(text)))
def median_iqr(s):
    q1, median, q3 = np.percentile(s, [25, 50, 75])
    return pd.Series({"median": median, "Q1": q1, "Q3": q3, "IQR": q3 - q1})

df["question_len"] = df["question"].apply(n_tokens)
df["context_len"] = df["context"].apply(n_tokens)

for split_df in [train_df, val_df]:
    split_df["question_len"] = split_df["question"].apply(n_tokens)
    split_df["context_len"] = split_df["context"].apply(n_tokens)


def corpus_stats_by_lang(split_df, split_name):
    rows = []
    for lang, g in split_df.groupby("lang"):
        q_stats = median_iqr(g["question_len"])
        c_stats = median_iqr(g["context_len"])
        rows.append({
            "split": split_name,
            "lang": lang,
            "n": len(g),
            "pct_answerable": g["answerable"].mean() * 100,
            "q_median_iqr": f'{q_stats["median"]:.1f} ({q_stats["IQR"]:.1f})',
            "c_median_iqr": f'{c_stats["median"]:.1f} ({c_stats["IQR"]:.1f})',
        })

    # "All languages" summary row for this split
    q_all = median_iqr(split_df["question_len"])
    c_all = median_iqr(split_df["context_len"])
    rows.append({
        "split": split_name,
        "lang": "All",
        "n": len(split_df),
        "pct_answerable": split_df["answerable"].mean() * 100,
        "q_median_iqr": f'{q_all["median"]:.1f} ({q_all["IQR"]:.1f})',
        "c_median_iqr": f'{c_all["median"]:.1f} ({c_all["IQR"]:.1f})',
    })
    return pd.DataFrame(rows)


combined_stats = pd.concat([
    corpus_stats_by_lang(train_df, "train"),
    corpus_stats_by_lang(val_df, "validation"),
], ignore_index=True)

print(combined_stats.to_string(index=False))


# ── Missing values and duplicates, reported per split (not per language) ──
for split_name, split_df in [("train", train_df), ("validation", val_df)]:
    print(f"\n{split_name}:")
    print(split_df.isna().sum())
    n_dup_rows = split_df.duplicated(subset=["question", "context"], keep="first").sum()
    print(f"redundant duplicate rows: {n_dup_rows}")

     split lang    n  pct_answerable q_median_iqr c_median_iqr
     train   ar 2558       90.031274    6.0 (2.0)  86.5 (67.0)
     train   ko 2422       97.398844    5.0 (2.0)  82.0 (63.0)
     train   te 1355       96.678967    5.0 (2.0)  77.0 (60.0)
     train  All 6335       94.269929    5.0 (3.0)  83.0 (64.0)
validation   ar  415       87.469880    6.0 (2.0)  84.0 (64.5)
validation   ko  356       94.662921    5.0 (2.0)  81.0 (71.2)
validation   te  384       75.781250    6.0 (2.0) 100.0 (63.2)
validation  All 1155       85.800866    5.0 (2.5)  89.0 (69.0)

train:
question            0
context             0
lang                0
answerable          0
answer_start        0
answer              0
answer_inlang    6285
split               0
question_len        0
context_len         0
dtype: int64
redundant duplicate rows: 10

validation:
question            0
context             0
lang                0
answerable          0
answer_start        0
answer              0
answer_inlang    1

In [90]:
#missing values
missing = df.isna().sum().to_frame("n_missing")
missing["pct_missing"] = (100 * missing["n_missing"]/len(df))
missing

#duplicates
dup_filter = df.duplicated(subset=["question", "context"], keep=False)
dup_amount = dup_filter.sum()
dup_groups = df[dup_filter].groupby(["question","context"]).ngroups
redudant_rows = df.duplicated(subset=["question", "context"], keep="first").sum()
print(dup_groups)
print(dup_amount)
print(redudant_rows)

10
20
10


# Tokenisation

In [91]:
#Limit reached so I will be using a local one 
from deep_translator import MyMemoryTranslator

try:
    r = MyMemoryTranslator(source="en-US", target="fr-FR").translate("hello")
    print("EN->FR:", r)
except Exception as e:
    print("failed:", type(e).__name__, e)

failed: TooManyRequests Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function


In [99]:
from deep_translator import GoogleTranslator

translator = GoogleTranslator(source="en", target="ar")
print(translator.translate("hello"))

TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function

In [92]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

LANG_MAP = {"ar": "arb_Arab","ko": "kor_Hang","te": "tel_Telu"}
def translate(token, lang):
    try:
        tok.src_lang = LANG_MAP[lang]
        inputs = tok(token, return_tensors="pt")
        output = model.generate(**inputs,forced_bos_token_id=tok.convert_tokens_to_ids("eng_Latn"))
        return tok.decode(output[0], skip_special_tokens=True)

    except Exception as e:
        print(f"failed: {token!r} ({lang}) -> {type(e).__name__}: {e}")
        return None

In [93]:
translation_cache = {}

def translate_cached(text, lang):
    key = (text, lang)
    if key in translation_cache:
        return translation_cache[key]
    result = translate(text, lang)
    translation_cache[key] = result
    return result

In [98]:
print(df.shape)

(7490, 10)


In [94]:
unique_questions = df[["question", "lang"]].drop_duplicates()
for q, lang in zip(unique_questions["question"], unique_questions["lang"]):
    translate_cached(q, lang) 

KeyboardInterrupt: 

In [ ]:
from collections import Counter, defaultdict

#We need to root out symbols so check for letter
has_letter = re.compile(r"\p{L}", flags=re.UNICODE)

def tokenize_words_xlmr(text):
    pieces = tok.tokenize(str(text))
    words, current = [], ""
    for p in pieces:
        if p.startswith("▁"):
            if current != "":
                words.append(current)
            current = p[1:]
        else:
            current += p
    if current != "":
        words.append(current)
    return [w.lower() for w in words if w != "" and has_letter.search(w)]

counters = defaultdict(Counter)
for lang, question in zip(df["lang"], df["question"]):
    counters[lang].update(tokenize_words_xlmr(question))

rows = [
    {"lang": lang, "token": token, "count": count}
    for lang, counter in counters.items()
    for token, count in counter.most_common(5)
]
top5_df = pd.DataFrame(rows)
top5_df = top5_df[top5_df["lang"].isin(["ar", "ko", "te"])]
top5_df["In English"] = [translate(token, lang) for token, lang in zip(top5_df["token"], top5_df["lang"])]
top5_df = top5_df[["lang", "token", "In English", "count"]]
top5_df

failed: '가장' (ko) -> RecursionError: maximum recursion depth exceededfailed: '가장' (ko) -> RecursionError: maximum recursion depth exceeded
failed: '무엇인가?' (ko) -> RecursionError: maximum recursion depth exceeded
failed: '언제' (ko) -> RecursionError: maximum recursion depth exceeded
failed: '몇' (ko) -> RecursionError: maximum recursion depth exceeded
failed: '어디인가?' (ko) -> RecursionError: maximum recursion depth exceeded
failed: 'في' (ar) -> RecursionError: maximum recursion depth exceeded
failed: 'من' (ar) -> RecursionError: maximum recursion depth exceeded
failed: 'متى' (ar) -> RecursionError: maximum recursion depth exceeded
failed: 'ما' (ar) -> RecursionError: maximum recursion depth exceeded
failed: 'هو' (ar) -> RecursionError: maximum recursion depth exceeded
failed: 'ఎవరు?' (te) -> RecursionError: maximum recursion depth exceeded
failed: 'ఎన్ని' (te) -> RecursionError: maximum recursion depth exceeded
failed: 'ఎప్పుడు' (te) -> RecursionError: maximum recursion depth exceeded
fail

,lang,token,In English,count
10,ko,가장,None,527
11,ko,무엇인가?,None,495
12,ko,언제,None,336
13,ko,몇,None,234
14,ko,어디인가?,None,228
25,ar,في,None,592
26,ar,من,None,584
27,ar,متى,None,535
28,ar,ما,None,441
29,ar,هو,None,349


In [10]:
#Task 3
def verify_answerable(df):
    checked = 0
    failures = []
    for i, row in enumerate(df):
        if row["answerable"]:
            checked+= 1
            context=row["context"]
            answer=row["answer"]
            start=row["answer_start"]

            expected=context[start:start+len(answer)]
            if expected!=answer:
                failures.append({
                    "index": i,
                    "answer_start": start,
                    "answer": answer,
                    "substring_found": expected,
                    "context": context})
    return checked,failures

#The pdf said to apply it to both splits
#we have to use ar,ko and te only, and they exist only in train and validation.
checked_train,failures_train=verify_answerable(train_df.to_dict("records"))
print(f"Checked {checked_train} answerable rows in the training set, found {len(failures_train)} failures")

checked_val,failures_val=verify_answerable(val_df.to_dict("records"))
print(f"Checked {checked_val} answerable rows in the validation set, found {len(failures_val)} failures")


Checked 13962 answerable rows in the training set, found 0 failures
Checked 2313 answerable rows in the validation set, found 0 failures


# i) Majority Baseline

In [11]:
majority_label = df["answerable"].value_counts().idxmax()
print("Majority class form training split: ", majority_label)

def majority_baseline(question=None, Context=None):
    return majority_label

Majority class form training split:  True


In [12]:
val_pred = [majority_baseline(q, c) for q, c in zip(val_df["question"], val_df["context"])]
acc_score = (pd.Series(val_pred).values == val_df["answerable"].values).mean()
print("Accuracy of Majority Baseline on Validation split: ", acc_score*100, "%")

Accuracy of Majority Baseline on Validation split:  76.81833277980738 %


## ii) Determinstic rule-based classifier baseline

- Raw tokenization overlapping between questions and contexts would never work(different languages)
- Whatever approach I take will struggle with the semintically equivalent words like created and invented, as we can't have semantic understanding without a model(unless I bring a synonym dictionary)
- What I am going to do is to basically translate the question using deep-translator's google translator, tokenize the question and the context, removing stop words, put into lower case, and possibly lemmatize, and then I will depend on lexical overlap between the tokens but not the pure thing(because of the semantically equivalent words problem), so I will try to map some question words to their expected answers using regex, and then I will use the overlap score between the question and the context to determine the answerability.
- What it can exploit is the samples that survives the translation, named entities, numbers, dates, and question-word typing. These will survive because proper nouns and numerals are often unchanged, and question-word categories are a small closed set you can pattern-match directly.
- What it can't exploit is the mistranslations, negations and basically anyhting requires semantic understanding.

In [63]:
import string

tokenizer= AutoTokenizer.from_pretrained("xlm-roberta-base")

STOPWORDS= {
    "the", "a", "an", "is", "are", "was", "were", "of", "in", "on", "at",
    "to", "for", "and", "or", "that", "this", "these", "those", "by", "with",
    "from", "as", "it", "its"
}

QUESTION_TYPE_PATTERNS= {
    "person":re.compile(r"\bwho\b", re.I),
    "date":re.compile(r"\bwhen\b", re.I),
    "location":re.compile(r"\bwhere\b", re.I),
    "number":re.compile(r"\bhow many\b|\bhow much\b", re.I),
}

ENTITY_PATTERNS= {
    "person":re.compile(r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+\b"),
    "date":re.compile(r"\b\d{1,2}(st|nd|rd|th)?\s\w+\s\d{4}\b|\b\d{4}\b|\b\w+\s\d{1,2},\s\d{4}\b"),
    "location":re.compile(r"\b[A-Z][a-z]+\b"),
    "number":re.compile(r"\b\d+([.,]\d+)?\b"),
}


def clean_tokens(text):
    tokens= tokenizer.tokenize(text.lower())
    tokens= [token for token in tokens if token and token not in string.punctuation]
    tokens= [token for token in tokens if token not in STOPWORDS]
    return set(tokens)


def lexical_overlap_score(question_en, context):
    q_tokens= clean_tokens(question_en)
    c_tokens= clean_tokens(context)
    if not q_tokens:
        return 0.0
    return len(q_tokens&c_tokens)/len(q_tokens)


def detect_question_type(question_en):
    for qtype, pattern in QUESTION_TYPE_PATTERNS.items():
        if pattern.search(question_en):
            return qtype
    return "other"


def context_has_type_match(context, qtype):
    if qtype == "other":
        return False
    pattern= ENTITY_PATTERNS.get(qtype)
    return bool(pattern.search(context)) if pattern else False


def predict_answerable(question, context, source_lang="auto"):
    question_en= translate(question, lang=source_lang)
    overlap= lexical_overlap_score(question_en, context)
    qtype= detect_question_type(question_en)
    type_match= context_has_type_match(context, qtype)
    prediction= (overlap>=0.1) or type_match
    return prediction

In [37]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
validation_df= val_df[val_df["lang"].isin(["ar", "ko", "te"])]

def evaluate_classifier(val_df):
    y_true, y_pred= [], []

    for row in val_df.to_dict("records"):
        pred = predict_answerable(
            row["question"], row["context"],
            source_lang=row["lang"],
        )
        y_true.append(bool(row["answerable"]))
        y_pred.append(pred)



    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    return metrics



metrics= evaluate_classifier(validation_df)
print(f"Accuracy:{metrics['accuracy']:.3f}")
print(f"Precision:{metrics['precision']:.3f}")
print(f"Recall:{metrics['recall']:.3f}")
print(f"F1:{metrics['f1']:.3f}")
print(f"Confusion matrix [[TN, FP], [FN, TP]]: {metrics['confusion_matrix']}")

#NOTE: Check the max seq length problem

Token indices sequence length is longer than the specified maximum sequence length for this model (792 > 512). Running this sequence through the model will result in indexing errors


failed: 'من هو رئيس الوزراء العراقي  الحالي؟' (ar) -> TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function


AttributeError: 'NoneType' object has no attribute 'lower'

# Week 37:

## BIO Labeling of the training split

In [64]:
#CODE CELL WAS IMPORTED FROM lab2.ipynb
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize_with_offsets(text):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    offsets = [(match.start(), match.end()) for match in matches]
    return tokens, offsets


def character_span_to_bio(context, answer_start=None, answer_text=""):
    tokens, offsets = tokenize_with_offsets(context)
    labels = ["O"] * len(tokens)

    if answer_start is None or answer_text == "":
        return tokens, offsets, labels

    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")

    covered = [
        index
        for index, (start, end) in enumerate(offsets)
        if start < answer_end and end > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")
    if offsets[covered[0]][0] != answer_start or offsets[covered[-1]][1] != answer_end:
        raise ValueError("The answer span does not align with token boundaries")

    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return tokens, offsets, labels


def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS"}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")

    answer_indices = [
        index for index, label in enumerate(labels) if label != "O"
    ]
    if not answer_indices:
        return None, ""

    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")

    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]

In [65]:
df = df.copy()

df = df.drop(columns=["tokens", "offsets", "labels", "alignable"], errors="ignore")

def try_bio(row):
    if not row["answerable"]:
        # Unanswerable: always succeeds, labels are all "O"
        tokens, offsets, labels = character_span_to_bio(row["context"], None, "")
        return pd.Series({"tokens": tokens, "offsets": offsets, "labels": labels, "alignable": True})
    try:
        tokens, offsets, labels = character_span_to_bio(
            row["context"], row["answer_start"], row["answer"]
        )
        return pd.Series({"tokens": tokens, "offsets": offsets, "labels": labels, "alignable": True})
    except ValueError:
        # Answer text doesn't align to token boundaries we flag it.
        return pd.Series({"tokens": None, "offsets": None, "labels": None, "alignable": False})

#Appending BIO columns to df
bio_results = df.apply(try_bio, axis=1)
df = pd.concat([df, bio_results], axis=1)

print(f"{(~df['alignable']).sum()} / {len(df)} rows fail alignment")
print(df.groupby("answerable")["alignable"].mean())


# Derive the clean training view by filtering out the failing rows
BIO_df = df[df["alignable"]].reset_index(drop=True)
dropped_df = df[~df["alignable"]]  # kept around for reporting


# Round-trip check: after BIO labeling, is the char length the same for tokens and labels?
mismatches = 0
for row in BIO_df.itertuples():
    if row.answerable:
        rt_start, rt_text = bio_to_character_span(row.context, row.offsets, row.labels)
        if rt_start != row.answer_start or rt_text != row.answer:
            mismatches += 1
    else:
        rt_start, rt_text = bio_to_character_span(row.context, row.offsets, row.labels)
        if rt_start is not None or rt_text != "":
            mismatches += 1

print(f"{mismatches} / {len(BIO_df)} rows failed round-trip check")

BIO_df[["context", "answerable", "tokens", "labels"]]

178 / 15343 rows fail alignment
answerable
False    1.000000
True     0.987251
Name: alignable, dtype: float64
0 / 15165 rows failed round-trip check


,context,answerable,tokens,labels
0,WikiLeaks () is an international non-profit or...,True,"[WikiLeaks, (, ), is, an, international, non, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
1,The war in Europe concluded with an invasion o...,True,"[The, war, in, Europe, concluded, with, an, in...","[O, O, O, O, O, O, O, O, O, B-ANS, O, O, O, O,..."
2,Same-sex marriage in the United States expande...,False,"[Same, -, sex, marriage, in, the, United, Stat...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,The exact number of Arab casualties is unknown...,True,"[The, exact, number, of, Arab, casualties, is,...","[O, O, O, O, O, O, O, B-ANS, O, O, O, O, O, O,..."
4,"As Thomas Hall (2000) notes, ""The Sung Empire ...",True,"[As, Thomas, Hall, (, 2000, ), notes, ,, "", Th...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
...,...,...,...,...
15160,"In February 2012, Somali government officials ...",True,"[In, February, 2012, ,, Somali, government, of...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
15161,The first earth tracks were created by humans ...,True,"[The, first, earth, tracks, were, created, by,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
15162,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",True,"[Abdel, Fattah, Saeed, Hussein, Khalil, El, -,...","[B-ANS, I-ANS, I-ANS, I-ANS, I-ANS, I-ANS, I-A..."
15163,Munich (; ; ) is the capital and most populous...,True,"[Munich, (, ;, ;, ), is, the, capital, and, mo...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


In [66]:
#A view of the rows where answerable = true but the length of the answer does not match
#with answer_start + len(answer)
dropped_df[["context", "answerable", "answer"]]

,context,answerable,answer
28,With the success of the Indian Navy's operatio...,True,India
68,"The Wright brothers, Orville (August 19, 1871 ...",True,America
113,The genocide in Bangladesh began on 26 March 1...,True,e
154,Putin studied Law at the Leningrad State Unive...,True,natoly Sobchak
282,"According to tradition, ""Vyasa"" is the compile...",True,Vyas
...,...,...,...
14777,"The tower is tall, about the same height as an...",True,ame height as an 81- building
14894,"According to the ""Oxford English Dictionary"", ...",True,"000,000"
14987,Her versatility is evident in Panchavataram on...,True,26 December 201
15000,Leonardo Del Vecchio started the company in 19...,True,Leonardo Del Vecchio started the company in 196


In [67]:
def run_bio_checks():
    checks_passed = 0
    checks_total = 0

    def check(name, condition):
        nonlocal checks_passed, checks_total
        checks_total += 1
        status = "PASS" if condition else "FAIL"
        if condition:
            checks_passed += 1
        print(f"[{status}] {name}")

    #Answer starting at character 0
    context = "Copenhagen is the capital of Denmark."
    answer_text = "Copenhagen"
    answer_start = 0
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    check("answer at char 0: first token is B-ANS", labels[0] == "B-ANS")
    check("answer at char 0: no other tokens are ANS", labels[1:] == ["O"] * (len(labels) - 1))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("answer at char 0: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Multi-token answer
    context = "The main character of Jujutsu Kaisen is Itadori Yuji."
    answer_text = "Itadori Yuji"
    answer_start = context.index(answer_text)
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    ans_indices = [i for i, l in enumerate(labels) if l != "O"]
    check("multi-token answer: spans exactly 2 tokens", len(ans_indices) == 2)
    check("multi-token answer: starts with B-ANS", labels[ans_indices[0]] == "B-ANS")
    check("multi-token answer: continues with I-ANS", all(labels[i] == "I-ANS" for i in ans_indices[1:]))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("multi-token answer: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Punctuation immediately adjacent to the answer (no whitespace separator)
    context = "She was born in Odense, Denmark."
    answer_text = "Odense"
    answer_start = context.index(answer_text)  # immediately followed by ','
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    ans_indices = [i for i, l in enumerate(labels) if l != "O"]
    check("punctuation-adjacent answer: exactly one ANS token (comma excluded)", len(ans_indices) == 1)
    check("punctuation-adjacent answer: comma token is 'O'", tokens[ans_indices[-1] + 1] == "," and labels[ans_indices[-1] + 1] == "O")
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("punctuation-adjacent answer: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Unanswerable example (no answer_start / answer_text)
    context = "This string contains no answer for the question."
    tokens, offsets, labels = character_span_to_bio(context, answer_start=None, answer_text="")
    check("unanswerable: all labels are 'O'", all(l == "O" for l in labels))
    check("unanswerable: tokens/offsets still populated", len(tokens) > 0 and len(offsets) == len(tokens))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("unanswerable: round-trips to (None, '')", (rt_start, rt_text) == (None, ""))

    print(f"\n{checks_passed}/{checks_total} checks passed")
    assert checks_passed == checks_total, "One or more BIO conversion checks failed"

run_bio_checks()

[PASS] answer at char 0: first token is B-ANS
[PASS] answer at char 0: no other tokens are ANS
[PASS] answer at char 0: round-trips correctly
[PASS] multi-token answer: spans exactly 2 tokens
[PASS] multi-token answer: starts with B-ANS
[PASS] multi-token answer: continues with I-ANS
[PASS] multi-token answer: round-trips correctly
[PASS] punctuation-adjacent answer: exactly one ANS token (comma excluded)
[PASS] punctuation-adjacent answer: comma token is 'O'
[PASS] punctuation-adjacent answer: round-trips correctly
[PASS] unanswerable: all labels are 'O'
[PASS] unanswerable: tokens/offsets still populated
[PASS] unanswerable: round-trips to (None, '')

13/13 checks passed


In [68]:
def token_features(tokens, index):
    word = tokens[index]
    features = {
        "bias": 1.0,
        "word.lower": word.lower(),
        "word.prefix2": word[:2].lower(),
        "word.suffix2": word[-2:].lower(),
        "word.suffix3": word[-3:].lower(),
        "word.istitle": word.istitle(),
        "word.isupper": word.isupper(),
        "word.isdigit": word.isdigit(),
        "contains_hyphen": "-" in word,
    }

    if index == 0:
        features["BOS"] = True
    else:
        previous = tokens[index - 1]
        features.update({
            "previous.lower": previous.lower(),
            "previous.istitle": previous.istitle(),
            "previous.isupper": previous.isupper(),
        })

    if index == len(tokens) - 1:
        features["EOS"] = True
    else:
        following = tokens[index + 1]
        features.update({
            "next.lower": following.lower(),
            "next.istitle": following.istitle(),
            "next.isupper": following.isupper(),
        })

    return features

In [69]:
def repair_bio_labels(labels):
    clean = ["O"] * len(labels)

    start = next((i for i, l in enumerate(labels) if l == "B-ANS"), None)
    if start is None:
        return clean   # model found no answer -> all O, which means empty span

    clean[start] = "B-ANS"
    i = start + 1
    while i < len(labels) and labels[i] == "I-ANS":
        clean[i] = "I-ANS"
        i += 1

    return clean

In [70]:
def token_features_qc(tokens, index, q_words):
    features = token_features(tokens, index)
    word = tokens[index]
    features["in_question"] = word.lower() in q_words         

    if index > 0:
        features["prev_in_question"] = tokens[index - 1].lower() in q_words
    if index < len(tokens) - 1:
        features["next_in_question"] = tokens[index + 1].lower() in q_words

    return features

In [71]:
def featurize_bio_df(df):
    features, labels, lengths = [], [], []
    for row in df.to_dict("records"):
        tokens = row["tokens"]
        q_words = clean_tokens(row.get("question_en", row.get("question")))

        lengths.append(len(tokens))
        features.extend(token_features_qc(tokens, i, q_words) for i in range(len(tokens)))
        labels.extend(row["labels"]) 
    return features, labels, lengths

In [72]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(BIO_df, test_size=0.2, random_state=42)
train_features, train_labels, train_lengths = featurize_bio_df(train_df)
val_features, val_labels, val_lengths = featurize_bio_df(val_df)

vectorizer = DictVectorizer(sparse=True)

def to_int32_indices(sparse_matrix):
    sparse_matrix.indices = sparse_matrix.indices.astype("int32")
    sparse_matrix.indptr = sparse_matrix.indptr.astype("int32")
    return sparse_matrix

X_train = to_int32_indices(vectorizer.fit_transform(train_features))
X_val = to_int32_indices(vectorizer.transform(val_features))

qc_classifier = SGDClassifier(loss="log_loss", alpha=1e-5, class_weight="balanced", max_iter=30, random_state=42,)
qc_classifier.fit(X_train, train_labels)

,"<a class=""param-doc-link"" style=""anchor-name: --doc-link-loss;"" rel=""noreferrer"" target=""_blank"" href=""https://scikit-learn.org/1.9/modules/generated/sklearn.linear_model.SGDClassifier.html#:~:text=loss,-%7B%27hinge%27%2C%20%27log_loss%27%2C%20%27modified_huber%27%2C%20%27squared_hinge%27%2C%20%20%20%20%20%20%20%20%27perceptron%27%2C%20%27squared_error%27%2C%20%27huber%27%2C%20%27epsilon_insensitive%27%2C%20%20%20%20%20%20%20%20%27squared_epsilon_insensitive%27%7D%2C%20default%3D%27hinge%27""> loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide<sgd_mathematical_formulation>` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'log_loss'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",1e-05
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method.Values must be in the range `[1, inf)`... versionadded:: 0.19",30
,"random_state random_state: int, RandomState instance, default=NoneUsed for shuffling the data, when ``shuffle`` is set to ``True``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.Integer values must be in the range `[0, 2**32 - 1]`.",42
,"class_weight class_weight: dict, {class_label: weight} or ""balanced"", default=NonePreset for the class_weight fit parameter.Weights associated with classes. If not given, all classesare supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, training will stopwhen (loss > best_loss - tol) for ``n_iter_no_change`` consecutiveepochs.Convergence is checked against the training loss or thevalidation loss depe

In [73]:
def predict_span_qc_classifier(row, vectorizer, classifier, q_words=None):
    tokens, offsets = row["tokens"], row["offsets"]
    question_en = row.get("question_en", row.get("question"))
    q_words = q_words or clean_tokens(question_en)

    feats = [token_features_qc(tokens, i, q_words) for i in range(len(tokens))]
    X = vectorizer.transform(feats)
    raw_labels = list(classifier.predict(X))

    clean_labels = repair_bio_labels(raw_labels)   # <-- new step
    return bio_to_character_span(row["context"], offsets, clean_labels)

In [76]:
import string

def lexical_overlap_span_baseline(question_en, context, min_overlap_tokens=1):
    tokens, offsets = tokenize_with_offsets(context)   # friend's tokenizer
    q_words = clean_tokens(question_en)                # your cleaning function

    overlap_flags = []
    for tok in tokens:
        cleaned = tok.lower()
        is_overlap = (
            cleaned not in string.punctuation
            and cleaned not in STOPWORDS
            and cleaned in q_words
        )
        overlap_flags.append(is_overlap)

    # group consecutive overlapping tokens into runs
    runs, run_start = [], None
    for i, flag in enumerate(overlap_flags):
        if flag and run_start is None:
            run_start = i
        elif not flag and run_start is not None:
            runs.append((run_start, i - 1))
            run_start = None
    if run_start is not None:
        runs.append((run_start, len(overlap_flags) - 1))

    if not runs:
        return None, ""   # nothing overlapped -> unanswerable

    best_start, best_end = max(runs, key=lambda r: r[1] - r[0])   # longest run wins
    if (best_end - best_start + 1) < min_overlap_tokens:
        return None, ""

    char_start = offsets[best_start][0]
    char_end = offsets[best_end][1]
    return char_start, context[char_start:char_end]

In [ ]:
def empty_baseline(question_en, context):
    return None, ""

In [77]:
print(train_df["lang"].value_counts())

lang
bn    2054
ar    2034
ko    1917
ja    1812
fi    1687
ru    1545
te    1083
Name: count, dtype: int64


In [ ]:
val_df["question_en"]= val_df.apply(lambda r: translate(r["question"], r["lang"]), axis=1)

In [35]:
for row in val_df.head(5).to_dict("records"):
    question_en = row.get("question_en", row.get("question"))
    print("Q:", question_en)
    print("  gold    :", row.get("answer_start"), repr(row.get("answer")))
    print("  empty   :", empty_baseline(question_en, row["context"]))
    print("  overlap :", lexical_overlap_span_baseline(question_en, row["context"]))
    print("  qc_model:", predict_span_qc_classifier(row, vectorizer, qc_classifier))
    print()

Q: キラル補助剤を開発したのは誰？
  gold    : 38 'E.J. Corey'
  empty   : (None, '')
  overlap : (None, '')
  qc_model: (0, 'Chiral')

Q: من توج ببطولة أوربا للأمم في عام 2004؟
  gold    : 480 'Greece'
  empty   : (None, '')
  overlap : (None, '')
  qc_model: (658, '1994')

Q: ما هي اسباب تخلي المغرب عن تنظيم كاس امم افريقيا عام 2015؟
  gold    : 108 'because of the Ebola virus epidemic in West Africa'
  empty   : (None, '')
  overlap : (None, '')
  qc_model: (None, '')

Q: 解析的整数論はいつ完成した？
  gold    : 220 '1837'
  empty   : (None, '')
  overlap : (None, '')
  qc_model: (None, '')

Q: భారతదేశంలో అతి పురాతన వృక్షం ఎక్కడ ఉంది?
  gold    : 4 'Great Banyan'
  empty   : (None, '')
  overlap : (None, '')
  qc_model: (339, '1925')

